# Verse Intelligence — Pipeline Analysis Notebook

This notebook walks through the full poetry NMT pipeline step-by-step with visualisations.

**Sections:**
1. Setup & Imports
2. Preprocessing Analysis
3. Neural Translation Demo
4. Embedding Visualisation
5. Evaluation Metrics
6. Sentiment Analysis
7. Batch Comparison — Multiple Poems
8. Language Pair Comparison

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML

# Plotting style
plt.style.use('dark_background')
PURPLE = '#7c3aed'
PINK   = '#ec4899'
GOLD   = '#f59e0b'
TEAL   = '#14b8a6'
GREEN  = '#10b981'
COLORS = [PURPLE, PINK, GOLD, TEAL, GREEN]

print('✓ Setup complete')

In [ ]:
# ── Cell 2: Preprocessing ─────────────────────────────────────────────────────
from pipeline.preprocessor import Preprocessor

poem_en = """Two roads diverged in a yellow wood,
And sorry I could not travel both
And be one traveler, long I stood
And looked down one as far as I could
To where it bent in the undergrowth;"""

preprocessor = Preprocessor()
result = preprocessor.process(poem_en)

print(f"Sentences  : {result['sentence_count']}")
print(f"Tokens     : {result['token_count']}")
print(f"Filtered   : {len(result['filtered_tokens'])}")
print(f"NER found  : {len(result['named_entities'])}")
print()
print("Top 15 tokens:", result['tokens'][:15])
print("Lemmas:",        result['lemmas'][:10])

In [ ]:
# ── Cell 3: POS Tag Distribution ──────────────────────────────────────────────
from collections import Counter

pos_counts = Counter(pt['tag'] for pt in result['pos_tags'])
tags   = list(pos_counts.keys())
counts = list(pos_counts.values())

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(tags, counts, color=PURPLE, alpha=0.85, edgecolor=PINK, linewidth=0.8)
ax.set_title('POS Tag Distribution', fontsize=14, color='white', pad=12)
ax.set_xlabel('POS Tag', color='#a89fd4')
ax.set_ylabel('Count',   color='#a89fd4')
ax.tick_params(colors='#a89fd4')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(count), ha='center', va='bottom', color='white', fontsize=9)
plt.tight_layout()
plt.savefig('pos_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: pos_distribution.png')

In [ ]:
# ── Cell 4: Translation ───────────────────────────────────────────────────────
from pipeline.translator import Translator

translator = Translator()

translations = {}
pairs = [("en","fr"),("en","de"),("en","es")]

for src, tgt in pairs:
    print(f"Translating EN → {tgt.upper()}...")
    t = translator.translate(poem_en, src_lang=src, tgt_lang=tgt)
    translations[tgt] = t
    print(f"  → {t[:80]}...\n")

print("✓ Translations complete")

In [ ]:
# ── Cell 5: Embedding Visualisation ───────────────────────────────────────────
from pipeline.embedder import Embedder

embedder = Embedder()
orig_emb = embedder.embed(poem_en)

trans_embs = {tgt: embedder.embed(txt) for tgt, txt in translations.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: First 60 dims comparison
ax = axes[0]
ax.plot(orig_emb[:60], color=PURPLE, label='Original (EN)', linewidth=1.5, alpha=0.9)
for i, (tgt, emb) in enumerate(trans_embs.items()):
    ax.plot(emb[:60], color=COLORS[i+1], label=f'Translated ({tgt.upper()})',
            linewidth=1.2, alpha=0.75)
ax.set_title('Embedding Comparison (dims 0–60)', color='white')
ax.set_xlabel('Dimension', color='#a89fd4')
ax.set_ylabel('Value', color='#a89fd4')
ax.legend(fontsize=8)
ax.tick_params(colors='#a89fd4')

# Plot 2: Cosine similarities
from sklearn.metrics.pairwise import cosine_similarity
cosines = {tgt: cosine_similarity([orig_emb], [emb])[0][0]
           for tgt, emb in trans_embs.items()}

ax2 = axes[1]
bars = ax2.barh(list(cosines.keys()), list(cosines.values()),
                color=[PINK, GOLD, TEAL], alpha=0.85, edgecolor='white', linewidth=0.5)
ax2.set_xlim(0, 1)
ax2.set_title('Cosine Similarity vs Original', color='white')
ax2.set_xlabel('Cosine Similarity', color='#a89fd4')
ax2.tick_params(colors='#a89fd4')
for bar, (tgt, val) in zip(bars, cosines.items()):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', color='white', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: embeddings.png')

In [ ]:
# ── Cell 6: Full Evaluation Metrics ───────────────────────────────────────────
from pipeline.evaluator import Evaluator

evaluator = Evaluator()
all_metrics = {}

for tgt, trans in translations.items():
    trans_emb = trans_embs[tgt]
    m = evaluator.evaluate(poem_en, trans, orig_emb, trans_emb)
    all_metrics[tgt] = m
    print(f"{tgt.upper()}:  cosine={m['cosine_similarity']:.4f}  "
          f"bleu={m['bleu_score']:.4f}  "
          f"bert_f1={m['bert_score']['f1']:.4f}  "
          f"drift={m['semantic_drift']:.4f}")

# Radar/bar chart
metric_names  = ['Cosine\nSimilarity', 'BLEU\nScore', 'BERTScore\nF1']
lang_labels   = list(all_metrics.keys())
metric_values = [
    [all_metrics[t]['cosine_similarity'] for t in lang_labels],
    [all_metrics[t]['bleu_score']        for t in lang_labels],
    [all_metrics[t]['bert_score']['f1']  for t in lang_labels],
]

x     = np.arange(len(metric_names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

for i, (lang, color) in enumerate(zip(lang_labels, [PURPLE, PINK, GOLD])):
    vals = [metric_values[j][i] for j in range(len(metric_names))]
    ax.bar(x + i*width, vals, width, label=lang.upper(), color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names, color='#a89fd4')
ax.set_ylim(0, 1)
ax.set_title('Evaluation Metrics by Target Language', color='white', fontsize=13)
ax.set_ylabel('Score', color='#a89fd4')
ax.legend()
ax.tick_params(colors='#a89fd4')
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: metrics_comparison.png')

In [ ]:
# ── Cell 7: Sentiment Analysis ────────────────────────────────────────────────
from pipeline.sentiment import SentimentAnalyzer

sa = SentimentAnalyzer()

poems = {
    "Frost (neutral)": poem_en,
    "Keats (positive)": "A thing of beauty is a joy for ever: its loveliness increases",
    "Thomas (intense)": "Do not go gentle into that good night, rage, rage against the dying of the light",
}

compounds = {}
for label, text in poems.items():
    r = sa.analyze(text)
    compounds[label] = r['compound']
    print(f"{label:30s} → {r['label']:10s}  compound={r['compound']:+.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
colors = [GREEN if v >= 0.05 else '#ef4444' if v <= -0.05 else GOLD
          for v in compounds.values()]
bars = ax.barh(list(compounds.keys()), list(compounds.values()),
               color=colors, alpha=0.85)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_xlim(-1, 1)
ax.set_title('VADER Sentiment Compound Scores', color='white', fontsize=13)
ax.set_xlabel('Compound Score (-1 = very negative, +1 = very positive)', color='#a89fd4')
ax.tick_params(colors='#a89fd4')
for bar, val in zip(bars, compounds.values()):
    ax.text(val + 0.02 if val >= 0 else val - 0.02,
            bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right',
            color='white', fontsize=9)
plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: sentiment_analysis.png')

In [ ]:
# ── Cell 8: Full Pipeline End-to-End ──────────────────────────────────────────
from pipeline.orchestrator import PoetryPipeline

pipeline = PoetryPipeline()
result = pipeline.run(
    poem=poem_en,
    src_lang='en',
    tgt_lang='fr',
    mode='literal',
    llm_api_key='',
    use_rag=False,
)

print("=" * 60)
print("ORIGINAL:")
print(poem_en)
print()
print("TRANSLATED (EN→FR):")
print(result['raw_translation'])
print()
print("METRICS:")
m = result['metrics']
print(f"  Cosine Similarity : {m['cosine_similarity']:.4f}")
print(f"  BLEU Score        : {m['bleu_score']:.4f}")
print(f"  BERTScore F1      : {m['bert_score']['f1']:.4f}")
print(f"  Semantic Drift    : {m['semantic_drift']:.4f}")
print()
print("SENTIMENT:")
s = result['sentiment']
print(f"  Original  : {s['original']['label']}  (compound={s['original']['compound']})")
print(f"  Translated: {s['translated']['label']}  (compound={s['translated']['compound']})")
print(f"  Drift     : {s['drift']['compound_drift']:.4f}  Preserved={s['drift']['preserved']}")
print("=" * 60)